In [2]:
import cv2
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog
from datetime import datetime
import matplotlib.pyplot as plt
import os
from common import get_image_properties

In [26]:
# Open file dialog to select the spreadsheet file
spreadsheet_path = filedialog.askopenfilename(
    title="Select the Spreadsheet File",
    filetypes=[("Excel files", "*.xlsx")]
)
sun_data = pd.read_excel(spreadsheet_path)
# Display the first few rows of the dataframe to understand its structure
sun_data.head()

,ID,Number of frames,Shape,Size,Location,Pathological diagnosis
0,1,527,Is,6,Cecum,Low-grade adenoma
1,2,1313,Is,18,Rectum,High-grade adenoma
2,3,292,IIa,3,Ascending colon,Low-grade adenoma
3,4,80,Is,4,Sigmoid colon,Low-grade adenoma
4,5,930,IIa,3,Transverse colon,Low-grade adenoma


In [28]:
# Replacing the header tags with their all-caps counterparts
sun_data.columns = [col.upper() for col in sun_data.columns]

In [27]:
# Extracting distinct values of pathological diagnosis
distinct_pathological_diagnoses = sun_data['PATHOLOGICAL DIAGNOSIS'].unique()
distinct_pathological_diagnoses.tolist()

['Low-grade adenoma',
 'High-grade adenoma',
 'Hyperplastic polyp',
 'Traditional serrated adenoma',
 'Sessile serrated lesion',
 'Invasive cancer (T1b)']

In [28]:
# Extracting distinct values of location
distinct_locations = sun_data['LOCATION'].unique()
distinct_locations.tolist()

['Cecum',
 'Rectum',
 'Ascending colon',
 'Sigmoid colon',
 'Transverse colon',
 'Descending colon']

In [29]:
'''
Shape data pre-processing
1. In cells where we find IIa(LST-NG), replacing that value with lla
2. Replacing values with the corresponding name
Ip	Pedunculated
Isp	Subpedunculated
Is	Sessile
IIa	Slightly elevated
'''

'\nShape data pre-processing\n1. In cells where we find IIa(LST-NG), replacing that value with lla\n2. Replacing values with the corresponding name\nIp\tPedunculated\nIsp\tSubpedunculated\nIs\tSessile\nIIa\tSlightly elevated\n'

In [33]:
# Replacing 'IIa(LST-NG)' with 'IIa' in the SHAPE column
sun_data['SHAPE'] = sun_data['SHAPE'].replace('IIa(LST-NG)', 'IIa')

# Verify if the replacement was successful
sun_data[sun_data['SHAPE'] == 'IIa(LST-NG)']

,ID,NUMBER OF FRAMES,SHAPE,SIZE,LOCATION,PATHOLOGICAL DIAGNOSIS


In [34]:
# Creating a mapping for SHAPE values to their corresponding names
shape_mapping = {
    'Ip': 'Pedunculated',
    'Isp': 'Subpedunculated',
    'Is': 'Sessile',
    'IIa': 'Slightly elevated'
}

# Replacing the SHAPE values with their corresponding names
sun_data['SHAPE'] = sun_data['SHAPE'].map(shape_mapping)

# Verifying the changes
sun_data.head()

,ID,NUMBER OF FRAMES,SHAPE,SIZE,LOCATION,PATHOLOGICAL DIAGNOSIS
0,1,527,Sessile,6,Cecum,Low-grade adenoma
1,2,1313,Sessile,18,Rectum,High-grade adenoma
2,3,292,Slightly elevated,3,Ascending colon,Low-grade adenoma
3,4,80,Sessile,4,Sigmoid colon,Low-grade adenoma
4,5,930,Slightly elevated,3,Transverse colon,Low-grade adenoma


In [32]:
'''
Create a new book in the spreadsheet that integrates the original spreadsheet information with the bounding box data from the text files
'''

'\nCreate a new book in the spreadsheet that integrates the original spreadsheet information with the bounding box data from the text files\n'

In [35]:
def extract_bounding_boxes(line):
    parts = line.strip().split(' ')
    filename = parts[0]
    bbox_strings = []

    # Iterate over the remaining parts of the line to get each bounding box
    for bbox_str in parts[1:]:
        # Split by comma and exclude the last element (class ID)
        bbox_values = bbox_str.split(',')[:-1]  # Excludes the class ID
        # Join the coordinates with commas and add to the list
        bbox_strings.append(','.join(bbox_values))

    # Join all bounding box strings; use a space only if there are multiple bounding boxes
    all_bboxes = ' '.join(bbox_strings) if len(bbox_strings) > 1 else ''.join(bbox_strings)
    return filename, all_bboxes

# Define the path where the case text files are located
txt_files_path = os.path.dirname(spreadsheet_path) + '/sundatabase_positive/annotation_txt'
# Initialize an empty list to store the new data rows
new_data_rows = []
# Iterate over each case in the original dataframe
for index, row in sun_data.iterrows():
    case_id = row['ID']
    shape = row['SHAPE']
    size = row['SIZE']
    location = row['LOCATION']
    diagnosis = row['PATHOLOGICAL DIAGNOSIS']
    
    # Construct the file path for the corresponding text file
    txt_file_path = os.path.join(txt_files_path, f'case{case_id}.txt')
    
    # Check if the text file exists
    if os.path.exists(txt_file_path):
        # Open and read the text file
        with open(txt_file_path, 'r') as file:
            for line in file:
                # We are going to exclude the last value referring to the class as it is 0 (polyp) in all the entries 
                
                filename, bounding_boxes = extract_bounding_boxes(line)
                             
                # Create a new row with combined information
                new_row = {
                    'ID': filename, 
                    'CASE ID': case_id, 
                    'SHAPE': shape, 
                    'SIZE': size, 
                    'LOCATION': location, 
                    'PATHOLOGICAL DIAGNOSIS': diagnosis, 
                    'BOUNDING BOX': bounding_boxes
                }
                new_data_rows.append(new_row)

# Convert the list of new rows into a DataFrame
new_data_df = pd.DataFrame(new_data_rows)
new_data_df.head()

,ID,CASE ID,SHAPE,SIZE,LOCATION,PATHOLOGICAL DIAGNOSIS,BOUNDING BOX
0,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"408,545,568,722"
1,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"388,549,561,729"
2,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"390,552,561,726"
3,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"390,552,561,730"
4,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"389,558,557,748"


In [36]:
# Convert the list of new rows into a DataFrame
new_data_df = pd.DataFrame(new_data_rows)

# Create a new book in the same Excel file
with pd.ExcelWriter(spreadsheet_path, engine='openpyxl', mode='a') as writer:
    new_data_df.to_excel(writer, sheet_name='Combined Data')

In [38]:
# Loading all books from the Excel file
multi_book_sheet = pd.read_excel(spreadsheet_path, sheet_name=None)
# Displaying the names of the books to identify the second book
multi_book_sheet.keys()
# Displaying the first few rows of the 'Combined Data' sheet
combined_data = multi_book_sheet['Combined Data']
combined_data.head()

,Unnamed: 0,ID,CASE ID,SHAPE,SIZE,LOCATION,PATHOLOGICAL DIAGNOSIS,BOUNDING BOX
0,0,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"408,545,568,722"
1,1,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"388,549,561,729"
2,2,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"390,552,561,726"
3,3,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"390,552,561,730"
4,4,case_M_20181001100941_0U62372100109341_1_005_0...,1,Sessile,6,Cecum,Low-grade adenoma,"389,558,557,748"


In [40]:
'''
Script that takes the list of images and produces a new spreadsheet with additional columns with properties of the images comprising the dataset.

For a preliminary analysis of a dataset intended for object detection it's important to focus on properties that provide insights into the dataset's characteristics and variability, without delving too deeply into complex feature extractions.
We look into the following properties:
- Image Dimensions: Width and height of each image. This helps to understand the size distribution of the images and whether any resizing or normalization is needed.
- Aspect Ratio: The ratio of width to height. This can be important for understanding how much variation there is in the shapes of the images.
- Color Distribution: Basic color analysis based on the average intensity per channel (Red, Green, Blue).
- Contrast and Brightness: Basic statistics on the brightness and contrast of the images can be useful to assess the need for pre-processing steps like contrast normalization.
- Presence of Blurriness assessed using the variance of the Laplacian.
'''

"\nScript that takes the list of images and produces a new spreadsheet with additional columns with properties of the images comprising the dataset.\n\nFor a preliminary analysis of a dataset intended for object detection it's important to focus on properties that provide insights into the dataset's characteristics and variability, without delving too deeply into complex feature extractions.\nWe look into the following properties:\n- Image Dimensions: Width and height of each image. This helps to understand the size distribution of the images and whether any resizing or normalization is needed.\n- Aspect Ratio: The ratio of width to height. This can be important for understanding how much variation there is in the shapes of the images.\n- Color Distribution: Basic color analysis based on the average intensity per channel (Red, Green, Blue).\n- Contrast and Brightness: Basic statistics on the brightness and contrast of the images can be useful to assess the need for pre-processing steps

In [39]:
base_folder = os.path.join(os.path.dirname(spreadsheet_path), 'sundatabase_positive')

# Initialize a list to store new rows with image properties
new_rows = []

# Iterate over each row in the Combined Data
for _, row in combined_data.iterrows():
    file_name = row['ID']
    case_id = row['CASE ID']
    
    # Construct the full path of the image
    image_path = os.path.join(base_folder, f'case{case_id}', file_name)
    
    # Analyze the image and get its properties
    if os.path.exists(image_path):
        width, height, aspect_ratio, avg_red, avg_green, avg_blue, brightness, contrast, blurriness = get_image_properties(image_path)
        
        # Add the new data to the row
        new_row = row.to_dict()
        new_row.update({
            'WIDTH': width,
            'HEIGHT': height,
            'ASPECT RATIO': aspect_ratio,
            'Avg Red Intensity': avg_red, 
            'Avg Green Intensity': avg_green,
            'Avg Blue Intensity': avg_blue,
            'BRIGHTNESS': brightness,
            'CONTRAST': contrast,
            'BLURRINESS': blurriness
        })
        new_rows.append(new_row)

# Convert the list of new rows into a DataFrame
new_data_df = pd.DataFrame(new_rows)

# Save the updated spreadsheets
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
new_spreadsheet_path = spreadsheet_path.replace('.xlsx', f'_{timestamp}.xlsx')

with pd.ExcelWriter(new_spreadsheet_path) as writer:
    # Save the new DataFrame to a new spreadsheet
    new_data_df.to_excel(writer, index=False)



In [ ]:
# Load the Excel file containing the annotations and image properties

In [3]:
spreadsheet_with_image_properties_path = filedialog.askopenfilename(
        title="Select the Spreadsheet File",
        filetypes=[("Excel files", "*.xlsx")]
    )

In [ ]:
# Load data

In [4]:
annotation_and_image_prop_set = pd.read_excel(spreadsheet_with_image_properties_path)
columns = annotation_and_image_prop_set.columns
num_entries = annotation_and_image_prop_set.shape[0]

columns, num_entries


(Index(['Unnamed: 0', 'ID', 'CASE ID', 'SHAPE', 'SIZE', 'LOCATION',
        'PATHOLOGICAL DIAGNOSIS', 'BOUNDING BOX', 'WIDTH', 'HEIGHT',
        'ASPECT RATIO', 'Avg Red Intensity', 'Avg Green Intensity',
        'Avg Blue Intensity', 'BRIGHTNESS', 'CONTRAST', 'BLURRINESS'],
       dtype='object'),
 49136)

In [5]:
# Number of unique polyps
unique_polyps = annotation_and_image_prop_set['CASE ID'].nunique()

unique_polyps

100

In [ ]:
# Distribution of polyp sizes. Figures

In [6]:
# Calculate the distribution of polyps by size for the training set
size_distribution = annotation_and_image_prop_set['SIZE'].value_counts().sort_index()

size_distribution

SIZE
2       922
3     11637
4      3769
5      7344
6      4243
7      3695
8      2791
10     2671
12      888
13     2946
15     6917
18     1313
Name: count, dtype: int64

In [ ]:
# Distribution of polyp sizes. Histogram

In [7]:
# Plot histograms for the distribution of polyps by size for both sets
plt.figure(figsize=(15, 6))
size_distribution.plot(kind='bar')
plt.title('Distribution of Polyps by Size')
plt.xlabel('Polyp Size')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

<Figure size 1500x600 with 1 Axes>

In [ ]:
# Distribution of polyp sizes. Descriptive statistics

In [7]:
# Extracting only the descriptive statistics for the polyp size in both sets
desc_stats_size = annotation_and_image_prop_set['SIZE'].describe()

desc_stats_size

count    49136.000000
mean         7.433552
std          4.565574
min          2.000000
25%          3.000000
50%          6.000000
75%         10.000000
max         18.000000
Name: SIZE, dtype: float64

In [ ]:
# Distribution of polyp sizes. Box plot

In [9]:
# Generate a combined vertical box plot for both sets
plt.figure(figsize=(8, 6))
annotation_and_image_prop_set['SIZE'].plot(kind='box')
plt.title('Box Plot of Polyp Sizes')
plt.ylabel('Size')
plt.grid(True)
plt.show()

<Figure size 800x600 with 1 Axes>

In [ ]:
# Distribution of polyp sizes based on the different ranges considered by POLAR's authors: Diminutive Polyps (1-5 mm), Small Polyps (6-9 mm), Larger Polyps (>=10 mm)

In [10]:
# Categorize polyp sizes into three groups
size_categories = pd.cut(annotation_and_image_prop_set['SIZE'], bins=[0, 5, 9, float('inf')], labels=['Diminutive (1-5 mm)', 'Small (6-9 mm)', 'Larger (>=10 mm)'])

# Calculate the distribution of polyps by these size categories
size_category_distribution = size_categories.value_counts().sort_index()

print(size_category_distribution)

# Generate a bar chart for the distribution of polyp sizes in the training and validation sets
size_category_distribution.plot(kind='bar', figsize=(10, 6))
plt.title('Distribution of Polyps by Size Category')
plt.xlabel('Polyp Size Category')
plt.ylabel('Number of Polyps')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.show()

SIZE
Diminutive (1-5 mm)    23672
Small (6-9 mm)         10729
Larger (>=10 mm)       14735
Name: count, dtype: int64


<Figure size 1000x600 with 1 Axes>

In [ ]:
# Distribution by pathology diagnosis

In [11]:
# Calculate the distribution of polyps by pathology diagnosis
pathology_distribution = annotation_and_image_prop_set['PATHOLOGICAL DIAGNOSIS'].value_counts()

# Generate histograms for the distribution of polyps by pathology diagnosis
plt.figure(figsize=(15, 6))
pathology_distribution.plot(kind='bar')
plt.title('Pathology Diagnosis Distribution')
plt.xlabel('Pathology Diagnosis')
plt.ylabel('Frequency')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

pathology_distribution

<Figure size 1500x600 with 1 Axes>

PATHOLOGICAL DIAGNOSIS
Low-grade adenoma               39834
High-grade adenoma               4111
Hyperplastic polyp               1644
Traditional serrated adenoma     1627
Sessile serrated lesion          1288
Invasive cancer (T1b)             632
Name: count, dtype: int64

In [ ]:
#Distribution by number of polyps (bounding boxes)

In [4]:
# Define a function to count the number of polyps based on bounding boxes
def count_polyps(bbox_str):
    # Split the bounding box string by spaces
    bbox_list = bbox_str.split(' ')
    # Return the number of bounding boxes, which corresponds to the number of polyps
    return len(bbox_list)

# Apply the function to the 'BOUNDING BOX' column
annotation_and_image_prop_set['Num Polyps'] = annotation_and_image_prop_set['BOUNDING BOX'].apply(count_polyps)

# Calculate the distribution of the number of polyps
polyp_distribution = annotation_and_image_prop_set['Num Polyps'].value_counts()

# Sort the distribution by the number of polyps
polyp_distribution_sorted = polyp_distribution.sort_index()

# Print the distribution
print(polyp_distribution_sorted)


Num Polyps
1    49134
2        1
6        1
Name: count, dtype: int64


49136

In [ ]:
# Polyp localization analysis. Descriptive statistics

In [5]:
# Function to extract coordinates from bounding box string
def extract_coordinates(bbox_str):
    # Split the string by spaces to separate each bounding box
    bboxes = bbox_str.split(' ')
    # Initialize lists to store coordinates
    coordinates = {'min_x': [], 'min_y': [], 'max_x': [], 'max_y': []}
    # Process each bounding box
    for bbox in bboxes:
        if bbox:
            mx, my, Mx, My = map(int, bbox.split(','))
            coordinates['min_x'].append(mx)
            coordinates['min_y'].append(my)
            coordinates['max_x'].append(Mx)
            coordinates['max_y'].append(My)
    return coordinates

# Apply the function and store the results in new DataFrame columns
coordinate_data = annotation_and_image_prop_set['BOUNDING BOX'].apply(extract_coordinates).apply(pd.Series)

# Calculate descriptive statistics for each coordinate
stats = {key: pd.Series([coord for sublist in coordinate_data[key] for coord in sublist]).describe() 
         for key in ['min_x', 'min_y', 'max_x', 'max_y']}

# Convert the stats dictionary to a DataFrame for better presentation
stats_df = pd.DataFrame(stats)

# Print the descriptive statistics
print(stats_df)

              min_x         min_y         max_x         max_y
count  49142.000000  49142.000000  49142.000000  49142.000000
mean     426.557649    460.771580    784.709352    816.518721
std      238.261076    219.235336    224.601263    204.508223
min        0.000000    -57.000000     90.000000     56.000000
25%      251.000000    308.000000    652.000000    698.000000
50%      428.000000    472.000000    779.000000    855.000000
75%      584.000000    626.000000    930.000000    980.000000
max     1198.000000   1034.000000   1239.000000   1173.000000 49136


In [8]:
# We identify the entries with negative coordinate values
# Function to check for negative coordinates in a bounding box string
def has_negative_coordinates(bbox_str):
    # Split the string by spaces to separate each bounding box
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        # Split the coordinates and convert them to integers
        coordinates = map(int, bbox.split(','))
        # Check if any coordinate is negative
        if any(coord < 0 for coord in coordinates):
            return True
    return False

# Apply the function to each bounding box and create a new column for flags
annotation_and_image_prop_set['Negative Coordinates'] = annotation_and_image_prop_set['BOUNDING BOX'].apply(has_negative_coordinates)

# Filter the DataFrame to get only the rows with negative coordinates
entries_with_negative_coordinates = annotation_and_image_prop_set[annotation_and_image_prop_set['Negative Coordinates']]
# Drop the temporary column 'Has Negative Coordinates'
annotation_and_image_prop_set.drop('Negative Coordinates', axis=1, inplace=True)

entries_with_negative_coordinates


,Unnamed: 0,ID,CASE ID,SHAPE,SIZE,LOCATION,PATHOLOGICAL DIAGNOSIS,BOUNDING BOX,WIDTH,HEIGHT,ASPECT RATIO,Avg Red Intensity,Avg Green Intensity,Avg Blue Intensity,BRIGHTNESS,CONTRAST,BLURRINESS,Negative Coordinates
14795,14795,case_M_20181018093806_0U62363101890605_1_006_0...,30,Slightly elevated,4,Sigmoid colon,Low-grade adenoma,"507,-56,783,96",1240,1080,1.148148,63.812150,96.675186,145.366716,101.951351,54.789463,708.246266,True
48903,48903,case_M_20190131094430_0U62372013151529_1_003_0...,99,Sessile,5,Sigmoid colon,Low-grade adenoma,"590,-57,1026,491",1240,1080,1.148148,62.003958,83.962124,141.272252,95.746111,59.471704,787.481014,True


In [ ]:
# Let's analyze visually what happens with images in the train set that have negative coordinates

In [14]:
# Define the path where the case text files are located
image_directory = os.path.dirname(spreadsheet_with_image_properties_path) + '/sundatabase_positive/'

# Function to draw bounding boxes on an image
def draw_bounding_boxes(row):
    # Construct the image path
    case_folder = f"case{row['CASE ID']}"
    image_path = image_directory + case_folder + '/' + row['ID']
    image = cv2.imread(image_path)

    if image is not None:
        # Convert the image from BGR to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Normalize pixel values if necessary
        if image_rgb.max() > 1.0:
            image_rgb = image_rgb / 255.0

        # Draw each bounding box on the image
        bboxes = row['BOUNDING BOX'].split(' ')
        for bbox in bboxes:
            x1, y1, x2, y2 = map(int, bbox.split(','))
            # Note: Matplotlib expects (R, G, B) order of colors
            cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0, 1, 0), 2)  # Green box

        # Display the image in RGB format
        plt.imshow(image_rgb)
        plt.title(f"Image ID: {row['ID']}")
        plt.show()
    else:
        print(f"Image not found: {image_path}")

for index, row in entries_with_negative_coordinates.iterrows():
    # Draw bounding boxes on the image
    draw_bounding_boxes(row)


<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [ ]:
# Replacing negative coordinate values with "0" seems to be safe 

In [15]:
# Function to replace negative coordinates with zero
def replace_negative_coordinates(bbox_str):
    modified_bboxes = []
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        coords = bbox.split(',')
        # Replace negative coordinates with zero
        modified_coords = [max(int(coord), 0) for coord in coords]
        modified_bbox = ','.join(map(str, modified_coords))
        modified_bboxes.append(modified_bbox)
    return ' '.join(modified_bboxes)

# Apply the function to each bounding box
annotation_and_image_prop_set['BOUNDING BOX'] = annotation_and_image_prop_set['BOUNDING BOX'].apply(replace_negative_coordinates)
# Save the updated spreadsheet
no_negative_suffix = "no-negative-coords"
spreadsheet_no_negative_coords = spreadsheet_with_image_properties_path.replace('.xlsx', f'_{no_negative_suffix}.xlsx')
# Save the updated DataFrame to a new Excel file
annotation_and_image_prop_set.to_excel(spreadsheet_no_negative_coords, index=False)


In [ ]:
# Are there bounding boxes that exceed the boundaries of the images? 

In [19]:
# Function to check if bounding boxes exceed image boundaries
def check_bbox_boundaries(bbox_str, img_width, img_height):
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        if bbox:
            x1, y1, x2, y2 = map(int, bbox.split(','))
            if x1 < 0 or y1 < 0 or x2 > img_width or y2 > img_height:
                return True
    return False

# List to store entries with bounding boxes that exceed boundaries
exceeding_entries = []

# Check each entry and store those that exceed boundaries
for index, row in annotation_and_image_prop_set.iterrows():
    if check_bbox_boundaries(row['BOUNDING BOX'], row['WIDTH'], row['HEIGHT']):
        exceeding_entries.append(row)

# Convert the list to a DataFrame (optional)
exceeding_entries_df = pd.DataFrame(exceeding_entries)

exceeding_entries_df


,Unnamed: 0,ID,CASE ID,SHAPE,SIZE,LOCATION,PATHOLOGICAL DIAGNOSIS,BOUNDING BOX,WIDTH,HEIGHT,ASPECT RATIO,Avg Red Intensity,Avg Green Intensity,Avg Blue Intensity,BRIGHTNESS,CONTRAST,BLURRINESS
793,793,case_M_20181003094031_0U62363100354631_1_001_0...,2,Sessile,18,Rectum,High-grade adenoma,"282,424,893,1097",1158,1008,1.148810,62.656827,74.048314,128.147698,88.284280,68.135901,371.615567
794,794,case_M_20181003094031_0U62363100354631_1_001_0...,2,Sessile,18,Rectum,High-grade adenoma,"284,427,892,1091",1158,1008,1.148810,62.565774,73.935907,128.126288,88.209323,68.231169,398.839111
795,795,case_M_20181003094031_0U62363100354631_1_001_0...,2,Sessile,18,Rectum,High-grade adenoma,"282,424,896,1092",1158,1008,1.148810,62.878021,73.791218,128.420574,88.363271,68.035780,334.020403
796,796,case_M_20181003094031_0U62363100354631_1_001_0...,2,Sessile,18,Rectum,High-grade adenoma,"265,425,853,1093",1158,1008,1.148810,62.537411,74.365070,128.872917,88.591799,68.497330,382.816948
797,797,case_M_20181003094031_0U62363100354631_1_001_0...,2,Sessile,18,Rectum,High-grade adenoma,"257,434,849,1088",1158,1008,1.148810,62.595718,74.218813,129.068298,88.627610,68.694337,402.657945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31657,31657,case_M_20181213131150_0U62365121395349_1_002_0...,66,Sessile,6,Sigmoid colon,Low-grade adenoma,"359,646,748,1123",1240,1080,1.148148,66.816537,83.150438,143.473260,97.813412,66.008693,706.508047
32179,32179,case_M_20181213131150_0U62365121395349_1_002_0...,66,Sessile,6,Sigmoid colon,Low-grade adenoma,"274,569,704,1081",1240,1080,1.148148,62.997054,78.938917,136.563938,92.833303,62.367707,1166.137715
32372,32372,case_M_20181213131150_0U62365121395349_1_002_0...,66,Sessile,6,Sigmoid colon,Low-grade adenoma,"555,660,918,1085",1240,1080,1.148148,68.987773,86.205177,151.915891,102.369614,64.158397,454.551963
35353,35353,case_M_20181217131216_0U62368121700016_1_002_0...,71,Sessile,4,Ascending colon,Low-grade adenoma,"187,732,661,1117",1240,1080,1.148148,41.766917,55.659106,94.065236,63.830420,60.584864,692.026389


In [ ]:
# Let's analyze visually what happens with those images

In [22]:
plot_limit = 5 
image_counter = 0
for index, row in exceeding_entries_df.iterrows():
    # Draw bounding boxes on the image
    draw_bounding_boxes(row)
    image_counter += 1
    if image_counter >= plot_limit:
        break

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

<Figure size 640x480 with 1 Axes>

In [23]:
# Function to adjust bounding boxes
def adjust_bounding_boxes(bbox_str, img_width, img_height):
    adjusted_bboxes = []
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        if bbox:
            x1, y1, x2, y2 = map(int, bbox.split(','))
            # Adjust x_max and y_max if they exceed image boundaries
            x2 = min(x2, img_width)
            y2 = min(y2, img_height)
            adjusted_bbox = ','.join(map(str, [x1, y1, x2, y2]))
            adjusted_bboxes.append(adjusted_bbox)
    return ' '.join(adjusted_bboxes)

# Apply the function to each bounding box
for index, row in annotation_and_image_prop_set.iterrows():
    img_width = row['WIDTH']
    img_height = row['HEIGHT']
    annotation_and_image_prop_set.at[index, 'BOUNDING BOX'] = adjust_bounding_boxes(row['BOUNDING BOX'], img_width, img_height)


# Save the updated spreadsheets
no_exceeding_coords_suffix = "no-exceeding-coords"
spreadsheet_no_negative_coords = spreadsheet_no_negative_coords.replace('.xlsx', f'_{no_exceeding_coords_suffix}.xlsx')
with pd.ExcelWriter(spreadsheet_no_negative_coords) as writer:
    annotation_and_image_prop_set.to_excel(writer, sheet_name='train_set', index=False)

print(f"Updated spreadsheet saved as {spreadsheet_no_negative_coords}")

Updated spreadsheet saved as /Users/irconde/Downloads/Databases/SUN/sun-annotations_20231209124744_no-negative-coords_no-exceeding-coords.xlsx


In [ ]:
# Polyp morphology analysis

In [26]:
# Analyzing the distribution of SHAPE in the validation set
polyp_morphology_distribution = annotation_and_image_prop_set['SHAPE'].value_counts()
polyp_morphology_distribution

SHAPE
Sessile              23154
Slightly elevated    17136
Pedunculated          4684
Subpedunculated       4162
Name: count, dtype: int64

In [ ]:
# Polyp location analysis

In [27]:
# Analyzing the distribution of LOCATION in the dataset
polyp_location_distribution = annotation_and_image_prop_set['LOCATION'].value_counts()
polyp_location_distribution

LOCATION
Sigmoid colon       16021
Ascending colon     14209
Transverse colon     6372
Rectum               5138
Cecum                4999
Descending colon     2397
Name: count, dtype: int64

In [ ]:
# Pathological diagnosis analysis

In [28]:
# Analyzing the distribution of PATHOLOGICAL DIAGNOSIS in the dataset
polyp_pathology_distribution = annotation_and_image_prop_set['PATHOLOGICAL DIAGNOSIS'].value_counts()
polyp_pathology_distribution

PATHOLOGICAL DIAGNOSIS
Low-grade adenoma               39834
High-grade adenoma               4111
Hyperplastic polyp               1644
Traditional serrated adenoma     1627
Sessile serrated lesion          1288
Invasive cancer (T1b)             632
Name: count, dtype: int64